In [2]:
from qdrant_manager import QdrantVectorDB


In [4]:
qdrant_vectordb = QdrantVectorDB()
client = qdrant_vectordb.client
collection_name = qdrant_vectordb.collection_name

INFO:root:✓ Kết nối đến Qdrant server tại localhost:6333 thành công
INFO:httpx:HTTP Request: GET http://localhost:6333/collections "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features "HTTP/1.1 200 OK"
INFO:root:✓ Tạo collection 'audio_features' thành công


INFO:httpx:HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


In [5]:
print(collection_name)

audio_features


In [6]:
from audio_processor import AudioPreprocessor
audio_preprocessor = AudioPreprocessor()

In [7]:
from pathlib import Path
import pandas as pd

# Lấy tất cả file audio từ thư mục
audio_dir = Path(r"D:\Project\HCSDLDPT\wind_instruments_1s")
all_audio_files = sorted(audio_dir.glob("*.wav"))[:15]  # Lấy 30 file đầu tiên

# Xử lý 30 file audio
results = []
for i, audio_file in enumerate(all_audio_files, 1):
    features = audio_preprocessor.preprocess(str(audio_file))
    if features is not None:
        results.append({
            'file_name': audio_file.name,
            'file_path': str(audio_file),
            'features': features,
            'features_shape': features.shape,
        })
        print(f"[{i}/30] ✓ {audio_file.name} - Shape: {features.shape}")
    else:
        print(f"[{i}/30] ✗ {audio_file.name} - Lỗi xử lý")

print(f"\n✓ Thành công: {len(results)}/30 file")

# Hiển thị kết quả
df_results = pd.DataFrame([
    {
        'STT': i+1,
        'File Name': r['file_name'],
        'Feature Dim': r['features_shape'][0]
    } 
    for i, r in enumerate(results)
])

print("\n" + "="*50)
print("KẾT QUẢ XỬ LÝ 30 FILE ÂM THANH ĐẦU TIÊN")
print("="*50)
print(df_results.to_string(index=False))
print(f"\nTổng cộng: {len(results)} file được xử lý thành công")
print(f"Mỗi file được biểu diễn bằng vector {results[0]['features_shape'][0]} chiều")

[1/30] ✓ 02f8e94d.wav - Shape: (68,)
[2/30] ✓ 0395ba61.wav - Shape: (68,)
[3/30] ✓ 04076350.wav - Shape: (68,)
[4/30] ✓ 05a3154f.wav - Shape: (68,)
[5/30] ✓ 05d0dfa7.wav - Shape: (68,)
[6/30] ✓ 05d54a68.wav - Shape: (68,)
[7/30] ✓ 060e13d6.wav - Shape: (68,)
[8/30] ✓ 064cfad3.wav - Shape: (68,)
[9/30] ✓ 06896eb5.wav - Shape: (68,)
[10/30] ✓ 06d164df.wav - Shape: (68,)
[11/30] ✓ 0720ef3b.wav - Shape: (68,)
[12/30] ✓ 07211dfa.wav - Shape: (68,)


d:\Project\HCSDLDPT\venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=512
  warnings.warn(
d:\Project\HCSDLDPT\venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1536
  warnings.warn(


[13/30] ✓ 07a99531.wav - Shape: (68,)
[14/30] ✓ 085652af.wav - Shape: (68,)
[15/30] ✓ 0879e568.wav - Shape: (68,)

✓ Thành công: 15/30 file

KẾT QUẢ XỬ LÝ 30 FILE ÂM THANH ĐẦU TIÊN
 STT    File Name  Feature Dim
   1 02f8e94d.wav           68
   2 0395ba61.wav           68
   3 04076350.wav           68
   4 05a3154f.wav           68
   5 05d0dfa7.wav           68
   6 05d54a68.wav           68
   7 060e13d6.wav           68
   8 064cfad3.wav           68
   9 06896eb5.wav           68
  10 06d164df.wav           68
  11 0720ef3b.wav           68
  12 07211dfa.wav           68
  13 07a99531.wav           68
  14 085652af.wav           68
  15 0879e568.wav           68

Tổng cộng: 15 file được xử lý thành công
Mỗi file được biểu diễn bằng vector 68 chiều


In [11]:
from qdrant_client.models import PointStruct

for i in range(len(results)):
    file_path = results[i]['file_path']
    features = results[i]['features']
    file_name = results[i]['file_name']
    metadata = {
        "file_name": file_name,
        "file_path": file_path,
    }

    operation_info = client.upsert(
        collection_name="audio_features",
        wait=True,
        points=[
            PointStruct(id = i, vector=features, payload= metadata ),
        ],
    )

    print(operation_info)

INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "

operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=2 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=3 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=4 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=5 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=6 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=7 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=8 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=9 status=<UpdateStatus.COMPLETED: 'completed'>


INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "

operation_id=10 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=11 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=12 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=13 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=14 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=15 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=16 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=17 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=18 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=19 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=20 status=<UpdateStatus.COMPLETED: 'completed'>


INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "

operation_id=21 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=22 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=23 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=24 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=25 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=26 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=27 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=28 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=29 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=30 status=<UpdateStatus.COMPLETED: 'completed'>


In [18]:
test_path = r"D:\Project\HCSDLDPT\wind_instruments_1s\213d2998.wav"
test_features = audio_preprocessor.preprocess(test_path)
search_result = client.query_points(
    collection_name="audio_features",
    query=test_features,
    with_payload=True,
    limit=3
).points

print(search_result)


INFO:httpx:HTTP Request: POST http://localhost:6333/collections/audio_features/points/query "HTTP/1.1 200 OK"


[ScoredPoint(id=29, version=30, score=0.92672205, payload={'file_name': '106f028c.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\106f028c.wav'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=21, version=22, score=0.9260608, payload={'file_name': '0cea4550.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\0cea4550.wav'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=25, version=26, score=0.91299057, payload={'file_name': '0f958ff0.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\0f958ff0.wav'}, vector=None, shard_key=None, order_value=None)]


In [19]:
for point in search_result:
    print(f"ID: {point.id}, Distance: {point.score}")
    print(f"Metadata: {point.payload}")

ID: 29, Distance: 0.92672205
Metadata: {'file_name': '106f028c.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\106f028c.wav'}
ID: 21, Distance: 0.9260608
Metadata: {'file_name': '0cea4550.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\0cea4550.wav'}
ID: 25, Distance: 0.91299057
Metadata: {'file_name': '0f958ff0.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\0f958ff0.wav'}


In [1]:
import os
import pandas as pd

# Load train.csv
df_train = pd.read_csv('train.csv')

# Get all .wav files from wind_instruments_1s folder
folder_path = 'wind_instruments_1s'
wind_files = set([f for f in os.listdir(folder_path) if f.endswith('.wav')])

# Filter train.csv to only include files in wind_instruments_1s
df_result = df_train[df_train['fname'].isin(wind_files)].reset_index(drop=True)

# Save to new CSV file
output_file = 'wind_instruments_1s.csv'
df_result.to_csv(output_file, index=False)

print(f"Created {output_file} with {len(df_result)} files")
print(f"Total files in wind_instruments_1s folder: {len(wind_files)}")
print(f"Matched files from train.csv: {len(df_result)}")
print(df_result.head())

Created wind_instruments_1s.csv with 510 files
Total files in wind_instruments_1s folder: 510
Matched files from train.csv: 510
          fname     label  manually_verified
0  02f8e94d.wav  Clarinet                  0
1  0395ba61.wav      Oboe                  0
2  04076350.wav  Clarinet                  0
3  05a3154f.wav      Oboe                  1
4  05d0dfa7.wav      Oboe                  0


In [8]:
import pandas as pd
from qdrant_client.models import PointStruct

# Load the wind_instruments_1s.csv to create a mapping of filename to label
wind_csv_path = 'wind_instruments_1s.csv'
wind_df = pd.read_csv(wind_csv_path)

# Create a dictionary mapping filename to label for quick lookup
fname_to_label = dict(zip(wind_df['fname'], wind_df['label']))

# Update the upsert loop with label from CSV
for i in range(len(results)):
    file_path = results[i]['file_path']
    features = results[i]['features']
    file_name = results[i]['file_name']
    
    # Get label from wind_instruments_1s.csv if the file exists in it
    label = fname_to_label.get(file_name, "unknown")
    
    metadata = {
        "file_name": file_name,
        "file_path": file_path,
        "label": label,
    }

    operation_info = client.upsert(
        collection_name="audio_features",
        wait=True,
        points=[
            PointStruct(id=i, vector=features, payload=metadata),
        ],
    )

    print(operation_info)

INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "

operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=2 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=3 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=4 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=5 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=6 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=7 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=8 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=9 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=10 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=11 status=<UpdateStatus.COMPLETED: 'completed'>


INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"


operation_id=12 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=13 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=14 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=15 status=<UpdateStatus.COMPLETED: 'completed'>


In [9]:
test_path = r"D:\Project\HCSDLDPT\wind_instruments_1s\213d2998.wav"
test_features = audio_preprocessor.preprocess(test_path)
search_result = client.query_points(
    collection_name="audio_features",
    query=test_features,
    with_payload=True,
    limit=3
).points

# print(search_result) 

INFO:httpx:HTTP Request: POST http://localhost:6333/collections/audio_features/points/query "HTTP/1.1 200 OK"


In [10]:
for point in search_result:
    print(f"ID: {point.id}, Distance: {point.score}")
    print(f"Metadata: {point.payload}")

ID: 7, Distance: 0.9068119
Metadata: {'file_name': '064cfad3.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\064cfad3.wav', 'label': 'Clarinet'}
ID: 3, Distance: 0.9050892
Metadata: {'file_name': '05a3154f.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\05a3154f.wav', 'label': 'Oboe'}
ID: 13, Distance: 0.90431905
Metadata: {'file_name': '085652af.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\085652af.wav', 'label': 'Oboe'}
